# Model 2 — Strategy Personalizer (SYNTHETIC)

Component 1 · Anchor AI (J26-IT-366). All results here are **SYNTHETIC**.

**Setup (run first).** Clones the C1 branch `dev/ravindu` of the team repo and installs Component 1's pinned requirements. If the repository is private, Colab asks you to sign in to GitHub (or use a personal access token in the URL). The cell also prints the Python version: C1 supports 3.11–3.13.

In [ ]:
import sys
print("Python", sys.version.split()[0])
!git clone --depth 1 -b dev/ravindu https://github.com/DulanjaR/AnchorAI.git
%cd AnchorAI
!pip install -q -r components/c1_disruption/requirements.txt

**Which data.** By default this uses the SYNTHETIC persona files in `components/c1_disruption/sim/data/`: personas, their special habits, 30 days of planned tasks, each day's schedule (free-time windows, work, travel, meals…) and each persona's disruption profile. Format: `components/c1_disruption/sim/README.md`. To use other files, change the paths below. Everything produced here is **SYNTHETIC**: it checks that the pipeline works, it is not a research finding.

In [ ]:
DATA = "components/c1_disruption/sim/data/"
PERSONAS = DATA + "SYNTHETIC_personas.csv"
HABITS = DATA + "SYNTHETIC_habits.csv"
TASKS = DATA + "SYNTHETIC_tasks_planned.csv"
SCHEDULE = DATA + "SYNTHETIC_schedule.csv"
PROFILE = DATA + "SYNTHETIC_disruption_profile.csv"
from components.c1_disruption.sim.personas import load_personas, load_schedule, load_tasks
personas = load_personas(PERSONAS, HABITS)
tasks = load_tasks(TASKS, personas)
schedule = load_schedule(SCHEDULE, personas)
from components.c1_disruption.sim.profile import load_profile
profile = load_profile(PROFILE, personas)
print(len(personas), "personas,", len(tasks), "planned tasks,", len(schedule), "scheduled days")

**Are the strategies really different?** The distinctness check plans every scenario with every strategy and reports how often each pair gives a different MVD. A pair that never differs would be merged or dropped.

In [ ]:
from components.c1_disruption.ml.bandit.actions import ACTIONS, distinctness
from components.c1_disruption.sim.generate import generate
from components.c1_disruption.sim.events import build_events
world = generate(personas, tasks, seed=11, schedule=schedule, profile=profile)
events, skipped = build_events(world, n_estimators=50)
print("strategies:", ACTIONS)
print(distinctness([e.engine_input for e in events]))
print(sum(s.kind == "no_effect" for s in skipped), "disruptions had no effect (they overlap no planned task)")
print(sum(s.kind == "cannot_plan" for s in skipped), "events the engine cannot plan (e.g. a None EF value, an open research decision)")

**Simulated acceptance.** Each SYNTHETIC student secretly prefers one strategy (`preferredStrategy`). We compare the fixed order, the per-student "most accepted" counter, random, and the pooled LinUCB bandit on the same events.

In [ ]:
from statistics import mean
from components.c1_disruption.sim.experiment import model2
curves = model2(world, events, seed=11)
for name, values in curves.items():
    print(f"{name:14} mean acceptance {mean(values) if values else float('nan'):.2f} over {len(values)} events")